**Reusing Pretrained Layer**

**What is Reusing Pretrained Layers?**

When an AI model (such as a neural network) is trained on a large dataset for a specific task (e.g., image classification on millions of samples), it learns general features that can be useful for other similar tasks. Instead of training a new model from scratch, we can reuse layers from a pretrained model for a new task.


---


**How are pretrained layers reused?**

1️⃣ Transferring layers to a new task: 🔀

We take some (or all) layers from a pretrained model and use them in a new model for a different task.

This saves training time and reduces the need for large datasets.

2️⃣ Freezing the layers: 🧊

Sometimes, we freeze the layers, meaning their weights remain unchanged during training on new data.

This is useful when the new task is very similar to the original task.

3️⃣ Fine-tuning the layers: 🎛️

Instead of freezing them, we can slightly update the weights (Fine-tuning) to adapt them to the new task.

This is useful when the new task is somewhat different from the original task.


---
**What are the benefits of reusing pretrained layers?**

✔ Faster training: Since the model doesn’t start from scratch, it leverages previously learned features.

✔ Reduced need for large datasets: Because pretrained layers already contain useful general features.

✔ Achieving high accuracy quickly: The model starts with prior knowledge instead of random initialization.


---
**Example:**

If we have a pretrained model for classifying animal images, we can reuse it to classify new types of animals instead of training a model from the beginning.

---
NOTE:

When performing **fine-tuning**, we make only **small adjustments** to the weights of the transferred layers. This is achieved by using a **low learning rate (LR)** to prevent drastic changes to the pretrained weights.

**The weight update formula:**

Wnew=Wold−LR⋅∇G

Where:

- Wnew  is the updated weight.
- Wold is the previous weight.
- LR (learning rate) controls the step size of updates.
- ∇G  is the gradient of the loss function.

A **small learning rate** ensures that the pretrained weights retain most of their learned knowledge while slightly adapting to the new dataset.




---


**"similar problem is enough for us to see effect" 💫**

**example :**

Using the **ByT5 model** for training on the **Arabic language** has shown effective results, sometimes even outperforming models specifically designed for Arabic. This can be attributed to the concept of **"reusing trained layers,"** where the model benefits from layers pretrained on multiple languages like
Spanish, among others. This approach not only saves time and resources but also allows the model to learn shared linguistic features across languages, enhancing its performance on Arabic language tasks.

https://huggingface.co/docs/transformers/en/model_doc/byt5 🤗

**example :**

T5: Text-to-Text Transfer Transformer

=> AraT5: Text-to-Text Transformers for Arabic Language Generation

AraT5 is a version of the T5 model specifically adapted for the Arabic language. However, unlike standard fine-tuning, it does not use the pre-trained weights from T5. Instead, only the architecture of the T5 model is used without leveraging the pre-trained weights. In this case, AraT5 is trained from scratch, using a large dataset of Arabic text, without using the data that Google used to train the original T5 model. Thus, the model is independently trained using only the new data and the base architecture

https://github.com/UBC-NLP/araT5

---
**Code & explanation**

split the fashion MNIST training set in two :

X_train_A: all images of all items except for sandals and shirts (classes 5 & 6)

X_train_B: a much smaller training set of just the first 200 images of sandals or shirts

The validation set and the test set ars also split this way, but without restricting the number of images

We will train a model on set A (classification task with 8 classes), and try to reuse it to tackle set B (binary classification).We hope to transfer a little bit of knowledge from task A to task B, since classes in ser A (sneakers, ankle boots, coats, t-shirt, etc) are somewhat similar to classes in set B (sandals and shirts)

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [2]:
# Download Fashion MNIST data
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalize data (convert values ​​to [0,1])
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

# Splitting data into training and validation
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# This code aims to split a dataset into two subsets based on target values ​​(classes) in y.

# Function to divide data into two groups A and B
def split_dataset(X, y):

  # Create an array that selects elements that are equal to 5 or 6
  y_5_or_6 = (y == 5) | (y == 6) # sandals or shirts

  # Create group A (data that is not 5 or 6)
  y_A = y[~y_5_or_6]

  # Edit the numbering to maintain the correct order.
  # Any category greater than 6 is decremented by 2, so that removing the categories does not cause gaps in the numbering
  y_A[y_A > 6] -= 2

  # Create group B (data belonging only to classes 5 and 6)
  y_B = (y[y_5_or_6]==6).astype(np.float32)

  # Return sets A and B
  return ((X[~y_5_or_6], y_A),
            (X[y_5_or_6], y_B))

# Apply the function to the training, validation, and test sets.
(X_train_A, y_train_A), (X_train_B, y_train_B) = split_dataset(X_train, y_train)
(X_valid_A, y_valid_A), (X_valid_B, y_valid_B) = split_dataset(X_valid, y_valid)
(X_test_A, y_test_A), (X_test_B, y_test_B) = split_dataset(X_test, y_test)

# Reduce the data size of set B to be only 200 in training
X_train_B = X_train_B[:200]
y_train_B = y_train_B[:200]

In [4]:
X_train_A.shape

(43986, 28, 28)

In [5]:
X_train_B.shape

(200, 28, 28)

In [6]:
y_train_A[:30]

array([4, 0, 5, 7, 7, 7, 4, 4, 3, 4, 0, 1, 6, 3, 4, 3, 2, 6, 5, 3, 4, 5,
       1, 3, 4, 2, 0, 6, 7, 1], dtype=uint8)

In [7]:
y_train_B[:30]

array([1., 1., 0., 0., 0., 0., 1., 1., 1., 0., 0., 1., 1., 0., 0., 0., 0.,
       0., 0., 1., 1., 0., 0., 1., 1., 0., 1., 1., 1., 1.], dtype=float32)

In [8]:
# Model A
# Set random values ​​to get repeatable results
tf.random.set_seed(42)
np.random.seed(42)

# Sequential model
model_A = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),# Flatten : Convert image to 1D matrix
    # 3 Hidden layers & output layer
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'), # Hidden Layer 1
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'), # Hidden Layer 2
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'), # Hidden Layer 3
    tf.keras.layers.Dense(8, activation='softmax') # output layer => 8 classes output , Softmax converts the outputs to probabilities whose sum is 1
])

# Preparing the model for training (Compilation)
model_A.compile(loss='sparse_categorical_crossentropy',
                optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
                metrics=['accuracy'])
# Model training
history = model_A.fit(X_train_A, y_train_A, epochs=20,validation_data=(X_valid_A, y_valid_A))

# Save the trained model for later use
model_A.save('model_A.keras')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
1375/1375 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.5307 - loss: 1.5598 - val_accuracy: 0.8281 - val_loss: 0.6290
Epoch 2/20
1375/1375 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8289 - loss: 0.5851 - val_accuracy: 0.8498 - val_loss: 0.4630
Epoch 3/20
1375/1375 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8517 - loss: 0.4580 - val_accuracy: 0.8647 - val_loss: 0.4054
Epoch 4/20
1375/1375 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8619 - loss: 0.4071 - val_accuracy: 0.8759 - val_loss: 0.3731
Epoch 5/20
1375/1375 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8700 - loss: 0.3780 - val_accuracy: 0.8837 - val_loss: 0.3528
Epoch 6/20
1375/1375 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8762 - loss: 0.3590 - val_accuracy: 0.8881 - val_loss: 0.3388
Epoch 7/20
1375/1375 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8803 - loss: 0.3453 - val_accuracy: 0.8919 - val_loss: 0.3282
Epoch 8/20
1375/1375 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8837 - loss: 0.3348 - 

In [9]:
# To test the performance of model model_A on test data (X_test_A, y_test_A) after training is complete
model_A.evaluate(X_test_A, y_test_A)

250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8954 - loss: 0.3072


[0.2994596064090729, 0.8973749876022339]

 How does evaluate work?

 The model passes the data (X_test_A) through the neural network

 It calculates the predicted values ​​and then compares them to the actual values ​​(y_test_A)

 It calculates a loss function based on the difference between the predicted and actual values

 It measures accuracy by the number of correct predictions

In [10]:
# Model B
# Set random values ​​to get repeatable results
tf.random.set_seed(42)
np.random.seed(42)

# Sequential model
model_B = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),# Flatten : Convert image to 1D matrix
    # 3 Hidden layers & output layer
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'), # Hidden Layer 1
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'), # Hidden Layer 2
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'), # Hidden Layer 3
    tf.keras.layers.Dense(1, activation='sigmoid') # output layer => 1 class output : binary classification, Sigmoid converts the output to values ​​between 0 and 1, making it suitable for binary classification (sandals vs. shirts)
])

# Preparing the model for training (Compilation)
model_B.compile(loss='binary_crossentropy',
                optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
                metrics=['accuracy'])
# Model training
history = model_B.fit(X_train_B, y_train_B, epochs=20,validation_data=(X_valid_B, y_valid_B))

# To test the performance of model model_B on test data (X_test_B, y_test_B) after training is complete
model_B.evaluate(X_test_B, y_test_B)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step - accuracy: 0.3597 - loss: 0.7420 - val_accuracy: 0.4249 - val_loss: 0.7178
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - accuracy: 0.4067 - loss: 0.7081 - val_accuracy: 0.4919 - val_loss: 0.6912
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4924 - loss: 0.6817 - val_accuracy: 0.6065 - val_loss: 0.6701
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.6076 - loss: 0.6609 - val_accuracy: 0.6744 - val_loss: 0.6526
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.6996 - loss: 0.6435 - val_accuracy: 0.6957 - val_loss: 0.6379
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.7268 - loss: 0.6287 - val_accuracy: 0.7140 - val_loss: 0.6252
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.7445 - loss: 0.6160 - val_accuracy: 0.7221 - val_loss: 0.6142
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7445 - loss: 0.6050 - val_accuracy: 0.7272 - val_loss: 0.6045


[0.5347093343734741, 0.7605000138282776]

In [11]:
model_B.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)                  │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 100)                 │          78,500 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 100)                 │          10,100 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 100)                 │          10,100 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 1)                   │             101 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 98,803 (385.95 KB)

 Trainable params: 98,801 (385.94 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [13]:
# This code reuses the pre-trained model_A to classify 8 classes,
# and then modifies it to create model_B_on_A to classify only 2 classes (sandals vs. shirts)

# Load the pre-trained model
model_A = keras.models.load_model('model_A.keras')

# A new model model_B_on_A is created using all layers in model_A except the last layer
# The goal is to retain the knowledge the model has learned about general features in images
model_B_on_A = keras.models.Sequential(model_A.layers[:-1])

# last layer of model_A replaced with a new layer has one neuron, this is suitable for binary classification
# sigmoid activation func. converts the output into a value between (0-1),representing the probability that the input belongs to one of the 2 classes (sandal | shirt)
model_B_on_A.add(keras.layers.Dense(1, activation='sigmoid'))

**Why this approach (Reusing Pretrained Layer) ?**

* Knowledge reuse: Because the hidden layers in model_A have already learned
useful features from the images.

* Accelerated training: Instead of training a new model from scratch, a previously trained model is modified to achieve faster and more accurate binary classification.



---

**Note :**

Note that model_B_on_A and model_A actually **share layers** now, so when we train one, it will **update both models**. If we want to **avoid that**, we need to **build model_B_on_A on top of a clone of model_A**


---

**Mutable and Immutable Data Types**

In programming, there are two types of data:

1️⃣ Mutable (changeable): It can be modified after it's created

2️⃣ Immutable (unchangeable): It can't be modified after it's created. If we want to change it, we must create a new copy of it

**example : array is mutable**


In [12]:
list_a = [1, 2, 3]
list_b = list_a # list_b refers to the same list in memory
list_b.append(4) # Add an element to the list
print(list_a) # Result: [1, 2, 3, 4] (The original list has been modified)

[1, 2, 3, 4]


In [13]:
list_a = [1, 2, 3]
list_b = list_a.copy() # Create a new copy
list_b.append(4)
print(list_a) # Result: [1, 2, 3] (the original list has not been modified)

[1, 2, 3]


**How does this relate to models in Keras?**

In the following code:

model_B_on_A = keras.models.Sequential(model_A.layers[:-1])

🔹 model_B_on_A doesn't get a separate copy of model_A's layers; it references the same original layers

🔹 So, when model_B_on_A is trained, model_A will also be updated, since they share the same layers



---

✅ To solve this problem, we need to **clone model_A to create an independent copy:** ⿻ 💪🏼

In [15]:
model_A_clone = keras.models.clone_model(model_A) # Architecture Clone
model_A_clone.set_weights(model_A.get_weights()) # Weights Clone
model_B_on_A = keras.models.Sequential(model_A_clone.layers[:-1])
model_B_on_A.add(keras.layers.Dense(1, activation='sigmoid'))

In [16]:
# Enable training for all layers except the last one
# By default, transferred layers are frozen (trainable = False)
# Setting trainable = True allows these layers to be fine-tuned on the new dataset
for layer in model_B_on_A.layers[:-1]:
  layer.trainable = True

model_B_on_A.compile(loss='binary_crossentropy',
                     optimizer=keras.optimizers.SGD(learning_rate=1e-1),
                     metrics=['accuracy'])
tf.random.set_seed(42)
np.random.seed(42)

history = model_B_on_A.fit(X_train_B, y_train_B, epochs=16,
                           validation_data=(X_valid_B, y_valid_B))

Epoch 1/16
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.9083 - loss: 0.2391 - val_accuracy: 0.9959 - val_loss: 0.0437
Epoch 2/16
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9894 - loss: 0.0371 - val_accuracy: 0.9939 - val_loss: 0.0340
Epoch 3/16
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9931 - loss: 0.0246 - val_accuracy: 0.9939 - val_loss: 0.0300
Epoch 4/16
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 0.0175 - val_accuracy: 0.9939 - val_loss: 0.0276
Epoch 5/16
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 1.0000 - loss: 0.0133 - val_accuracy: 0.9939 - val_loss: 0.0261
Epoch 6/16
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 1.0000 - loss: 0.0106 - val_accuracy: 0.9949 - val_loss: 0.0250
Epoch 7/16
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 1.0000 - loss: 0.0088 - val_accuracy: 0.9949 - val_loss: 0.0243
Epoch 8/16
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 0.0075 - val_accuracy: 0.9949 - val_loss: 0.0237




---


The comparison will be made on test dataset


In [17]:
model_B.evaluate(X_test_B, y_test_B)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7582 - loss: 0.5346


[0.5347093343734741, 0.7605000138282776]

In [18]:
model_B_on_A.evaluate(X_test_B, y_test_B)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9984 - loss: 0.0098


[0.009777504950761795, 0.9980000257492065]

It is **proven** that using a **pretrained model** improves **accuracy** 🕵🧾✔️

Model B, trained only on two classes, achieved 0.7582 accuracy, while model_B_on_A, which leveraged pretrained layers from A, reached 0.9984 accuracy due to transferred knowledge



---

**Real World Example**

The BYT5 model is pretrained on around 100 languages, and its primary task is to predict the missing word in a sentence to make the meaning correct. This model was used for the task of Fine-Tashkeel, which involves improving Arabic text diacritization, i.e., adding the correct diacritical marks (harakat) to Arabic text

The idea of Fine-Tashkeel is to fine-tune the BYT5 model so that it can perform Arabic diacritization with high accuracy.
The fine-tuned BYT5 model achieved an impressive 99.3% accuracy, significantly outperforming previous models that were built from scratch using the same dataset

**BYT5**

https://huggingface.co/docs/transformers/model_doc/byt5

**Fine-Tashkeel**

https://arxiv.org/abs/2303.14588




